In [ ]:
!pip install streamlit pyngrok PyJWT

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 58.5 MB/s eta 0:00:00


In [ ]:
import secrets
print(secrets.token_hex(32))

656a14b8cc88dae05337636593af9a3024068249b9029aa35ec7853c9db6d7ba


In [6]:
from google.colab import userdata

JWT_SECRET = userdata.get("JWT_SECRET")
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
EMAIL_ADDRESS = userdata.get("EMAIL_ADDRESS")
EMAIL_PASSWORD = userdata.get("EMAIL_PASSWORD")

print("JWT_SECRET:", "OK" if JWT_SECRET else "MISSING")
print("NGROK_AUTHTOKEN:", "OK" if NGROK_AUTHTOKEN else "MISSING")
print("EMAIL_ADDRESS:", "OK" if EMAIL_ADDRESS else "MISSING")
print("EMAIL_PASSWORD:", "OK" if EMAIL_PASSWORD else "MISSING")

JWT_SECRET: OK
NGROK_AUTHTOKEN: OK
EMAIL_ADDRESS: OK
EMAIL_PASSWORD: OK


In [7]:
!pip install streamlit pyngrok PyJWT

In [ ]:
%%writefile app.py
import streamlit as st

st.set_page_config(
    page_title="User Authentication",
    page_icon="🔐",
    layout="centered"
)

st.title("🔐 User Authentication System")
st.write("Infosys Springboard Internship 7.0")
st.write("Batch 4 — Milestone 1")

st.info("Your authentication application is being set up.")

page = st.sidebar.selectbox(
    "Choose Page",
    ["Login", "Signup", "Forgot Password", "Admin Login"]
)

st.subheader(page)

if page == "Login":
    st.write("Login page will be implemented here.")

elif page == "Signup":
    st.write("Signup page will be implemented here.")

elif page == "Forgot Password":
    st.write("Password recovery will be implemented here.")

elif page == "Admin Login":
    st.write("Admin login will be implemented here.")

Writing app.py


In [ ]:
from google.colab import userdata
from pyngrok import ngrok
import subprocess
import time

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))

process = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.address", "0.0.0.0"
])

time.sleep(5)

public_url = ngrok.connect(8501)
print("Open your application here:")
print(public_url)

Open your application here:
NgrokTunnel: "https://pregame-study-contend.ngrok-free.dev" -> "http://localhost:8501"


In [8]:

%%writefile app.py
import os
import sqlite3
import hashlib
import hmac
import jwt
import streamlit as st
from datetime import datetime, timedelta, timezone

DB_NAME = "users.db"
JWT_SECRET = os.environ.get("JWT_SECRET")

if not JWT_SECRET:
    st.error("JWT_SECRET is missing. Restart the app using the setup cell.")
    st.stop()

st.set_page_config(
    page_title="Infosys Authentication System",
    page_icon="🔐",
    layout="centered"
)

def connect_db():
    return sqlite3.connect(DB_NAME)

def create_table():
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                email TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                salt TEXT NOT NULL
            )
        """)

def hash_password(password, salt):
    return hashlib.pbkdf2_hmac(
        "sha256",
        password.encode(),
        bytes.fromhex(salt),
        200000
    ).hex()

def create_token(username):
    payload = {
        "username": username,
        "exp": datetime.now(timezone.utc) + timedelta(hours=1)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def get_logged_in_user():
    token = st.session_state.get("jwt_token")
    if not token:
        return None
    try:
        payload = jwt.decode(
            token,
            JWT_SECRET,
            algorithms=["HS256"]
        )
        return payload["username"]
    except jwt.InvalidTokenError:
        st.session_state.pop("jwt_token", None)
        return None

create_table()

st.title("🔐 User Authentication System")
st.caption("Infosys Springboard Internship 7.0 | Batch 4 | Milestone 1")

logged_in_user = get_logged_in_user()

if logged_in_user:
    st.success(f"Welcome, {logged_in_user}!")
    st.subheader("User Dashboard")
    st.write("You have successfully logged in.")
    st.write("Your session is protected using a JWT token.")

    if st.button("Logout"):
        st.session_state.pop("jwt_token", None)
        st.rerun()

else:
    page = st.sidebar.selectbox(
        "Choose Page",
        ["Login", "Signup", "Forgot Password"]
    )

    if page == "Signup":
        st.subheader("Create an Account")

        username = st.text_input("Username")
        email = st.text_input("Email Address")
        password = st.text_input("Password", type="password")
        confirm_password = st.text_input(
            "Confirm Password",
            type="password"
        )

        if st.button("Sign Up"):
            username = username.strip()
            email = email.strip().lower()

            if not username or not email or not password:
                st.error("Please fill in all fields.")
            elif "@" not in email or "." not in email.split("@")[-1]:
                st.error("Please enter a valid email address.")
            elif len(password) < 8:
                st.error("Password must contain at least 8 characters.")
            elif password != confirm_password:
                st.error("Passwords do not match.")
            else:
                salt = os.urandom(16).hex()
                password_hash = hash_password(password, salt)

                try:
                    with connect_db() as conn:
                        conn.execute(
                            """INSERT INTO users
                            (username, email, password_hash, salt)
                            VALUES (?, ?, ?, ?)""",
                            (username, email, password_hash, salt)
                        )
                    st.success("Account created! Select Login from the sidebar.")
                except sqlite3.IntegrityError:
                    st.error("That username or email is already registered.")

    elif page == "Login":
        st.subheader("Login to Your Account")

        username = st.text_input("Username")
        password = st.text_input("Password", type="password")

        if st.button("Login"):
            with connect_db() as conn:
                user = conn.execute(
                    """SELECT password_hash, salt FROM users
                    WHERE username = ?""",
                    (username.strip(),)
                ).fetchone()

            if user:
                entered_hash = hash_password(password, user[1])

                if hmac.compare_digest(entered_hash, user[0]):
                    st.session_state["jwt_token"] = create_token(
                        username.strip()
                    )
                    st.rerun()
                else:
                    st.error("Incorrect username or password.")
            else:
                st.error("Incorrect username or password.")

    elif page == "Forgot Password":
        st.subheader("Forgot Password")
        st.info(
            "Password recovery using email OTP will be added in the next step."
        )


Overwriting app.py


In [ ]:
%%writefile app.py
import os
import sqlite3
import hashlib
import hmac
import jwt
import random
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import streamlit as st
from datetime import datetime, timedelta, timezone

DB_NAME = "users.db"
JWT_SECRET = os.environ.get("JWT_SECRET")
EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD")

if not JWT_SECRET:
    st.error("JWT_SECRET is missing. Restart the app using the setup cell.")
    st.stop()

st.set_page_config(
    page_title="Infosys Authentication System",
    page_icon=" 🔐",
    layout="centered"
)

def connect_db():
    return sqlite3.connect(DB_NAME)

def create_tables():
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                email TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                salt TEXT NOT NULL
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS reset_otps (
                email TEXT PRIMARY KEY,
                otp TEXT NOT NULL,
                expires_at TIMESTAMP NOT NULL
            )
        """)

def hash_password(password, salt):
    return hashlib.pbkdf2_hmac(
        "sha256",
        password.encode(),
        bytes.fromhex(salt),
        200000
    ).hex()

def create_token(username):
    payload = {
        "username": username,
        "exp": datetime.now(timezone.utc) + timedelta(hours=1)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def get_logged_in_user():
    token = st.session_state.get("jwt_token")
    if not token:
        return None
    try:
        payload = jwt.decode(
            token,
            JWT_SECRET,
            algorithms=["HS256"]
        )
        return payload["username"]
    except jwt.InvalidTokenError:
        st.session_state.pop("jwt_token", None)
        return None

def send_otp_email(receiver_email, otp):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        st.error("Email sender configurations are missing.")
        return False

    sender_email = EMAIL_ADDRESS
    password = EMAIL_PASSWORD

    message = MIMEMultipart("alternative")
    message["Subject"] = "Your Password Reset OTP Code"
    message["From"] = sender_email
    message["To"] = receiver_email

    text = f"Your verification code is: {otp}. This code is valid for 10 minutes."
    html = f"""\
    <html>
      <body>
        <p>Hello,<br><br>
           You requested a password reset. Use the verification code below to reset your password:<br>
           <h2 style='color:#4CAF50;'>{otp}</h2><br>
           This code will expire in 10 minutes.
        </p>
      </body>
    </html>
    """

    part1 = MIMEText(text, "plain")
    part2 = MIMEText(html, "html")
    message.attach(part1)
    message.attach(part2)

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(sender_email, password)
            server.sendmail(sender_email, receiver_email, message.as_string())
        return True
    except Exception as e:
        st.error(f"Failed to send email: {e}")
        return False

create_tables()

st.title(" 🔐 User Authentication System")
st.caption("Infosys Springboard Internship 7.0 | Batch 4 | Milestone 1")

logged_in_user = get_logged_in_user()

if logged_in_user:
    st.success(f"Welcome, {logged_in_user}!")
    st.subheader("User Dashboard")
    st.write("You have successfully logged in.")
    st.write("Your session is protected using a JWT token.")

    if st.button("Logout"):
        st.session_state.pop("jwt_token", None)
        st.rerun()

else:
    page = st.sidebar.selectbox(
        "Choose Page",
        ["Login", "Signup", "Forgot Password"]
    )

    if page == "Signup":
        st.subheader("Create an Account")

        username = st.text_input("Username")
        email = st.text_input("Email Address")
        password = st.text_input("Password", type="password")
        confirm_password = st.text_input(
            "Confirm Password",
            type="password"
        )

        if st.button("Sign Up"):
            username = username.strip()
            email = email.strip().lower()

            if not username or not email or not password:
                st.error("Please fill in all fields.")
            elif "@" not in email or "." not in email.split("@")[-1]:
                st.error("Please enter a valid email address.")
            elif len(password) < 8:
                st.error("Password must contain at least 8 characters.")
            elif password != confirm_password:
                st.error("Passwords do not match.")
            else:
                salt = os.urandom(16).hex()
                password_hash = hash_password(password, salt)

                try:
                    with connect_db() as conn:
                        conn.execute(
                            """INSERT INTO users
                            (username, email, password_hash, salt)
                            VALUES (?, ?, ?, ?)""",
                            (username, email, password_hash, salt)
                        )
                    st.success("Account created! Select Login from the sidebar.")
                except sqlite3.IntegrityError:
                    st.error("That username or email is already registered.")

    elif page == "Login":
        st.subheader("Login to Your Account")

        username = st.text_input("Username")
        password = st.text_input("Password", type="password")

        if st.button("Login"):
            with connect_db() as conn:
                user = conn.execute(
                    """SELECT password_hash, salt FROM users
                    WHERE username = ?""",
                    (username.strip(),)
                ).fetchone()

            if user:
                entered_hash = hash_password(password, user[1])

                if hmac.compare_digest(entered_hash, user[0]):
                    st.session_state["jwt_token"] = create_token(
                        username.strip()
                    )
                    st.rerun()
                else:
                    st.error("Incorrect username or password.")
            else:
                st.error("Incorrect username or password.")

    elif page == "Forgot Password":
        st.subheader("Forgot Password Recovery")

        if "reset_step" not in st.session_state:
            st.session_state["reset_step"] = "request"
        if "reset_email" not in st.session_state:
            st.session_state["reset_email"] = ""

        if st.session_state["reset_step"] == "request":
            email = st.text_input("Enter your registered Email Address")
            if st.button("Send OTP"):
                email = email.strip().lower()
                if not email:
                    st.error("Please enter your email address.")
                else:
                    with connect_db() as conn:
                        user = conn.execute(
                            "SELECT id FROM users WHERE email = ?", (email,)
                        ).fetchone()

                    if user:
                        otp = str(random.randint(100000, 999999))
                        expires_at = (datetime.now() + timedelta(minutes=10)).strftime("%Y-%m-%d %H:%M:%S")

                        with connect_db() as conn:
                            conn.execute(
                                """INSERT OR REPLACE INTO reset_otps (email, otp, expires_at)
                                   VALUES (?, ?, ?)""", (email, otp, expires_at)
                            )

                        if send_otp_email(email, otp):
                            st.session_state["reset_email"] = email
                            st.session_state["reset_step"] = "verify"
                            st.success("An OTP has been sent to your email. Please check your inbox!")
                            st.rerun()
                    else:
                        st.error("No account found with this email address.")

        elif st.session_state["reset_step"] == "verify":
            st.info(f"OTP sent to: {st.session_state['reset_email']}")
            entered_otp = st.text_input("Enter OTP Code", max_chars=6)
            new_password = st.text_input("Enter New Password", type="password")
            confirm_new_password = st.text_input("Confirm New Password", type="password")

            col1, col2 = st.columns(2)
            with col1:
                if st.button("Verify & Reset Password"):
                    entered_otp = entered_otp.strip()
                    if not entered_otp or not new_password or not confirm_new_password:
                        st.error("Please fill out all fields.")
                    elif len(new_password) < 8:
                        st.error("Password must be at least 8 characters long.")
                    elif new_password != confirm_new_password:
                        st.error("Passwords do not match.")
                    else:
                        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                        with connect_db() as conn:
                            otp_record = conn.execute(
                                """SELECT otp, expires_at FROM reset_otps
                                   WHERE email = ? AND expires_at > ?""",
                                (st.session_state["reset_email"], current_time)
                            ).fetchone()

                        if otp_record and otp_record[0] == entered_otp:
                            # OTP is valid, proceed with password update
                            new_salt = os.urandom(16).hex()
                            new_hash = hash_password(new_password, new_salt)

                            with connect_db() as conn:
                                conn.execute(
                                    "UPDATE users SET password_hash = ?, salt = ? WHERE email = ?",
                                    (new_hash, new_salt, st.session_state["reset_email"])
                                )
                                conn.execute("DELETE FROM reset_otps WHERE email = ?", (st.session_state["reset_email"],))

                            st.success("Your password has been reset successfully! You can now log in.")
                            st.session_state["reset_step"] = "request"
                            st.session_state["reset_email"] = ""
                        else:
                            st.error("Invalid or expired OTP code.")
            with col2:
                if st.button("Cancel / Go Back"):
                    st.session_state["reset_step"] = "request"
                    st.session_state["reset_email"] = ""
                    st.rerun()


In [6]:

from google.colab import userdata
from pyngrok import ngrok
import subprocess
import os
import time

# Stop the previous Streamlit app if it exists
try:
    process.terminate()
    process.wait(timeout=5)
except Exception:
    pass

# Close old ngrok tunnels
ngrok.kill()

# Load secrets without printing their values
env = os.environ.copy()
env["JWT_SECRET"] = userdata.get("JWT_SECRET")
env["EMAIL_ADDRESS"] = userdata.get("EMAIL_ADDRESS")
env["EMAIL_PASSWORD"] = userdata.get("EMAIL_PASSWORD")
env["ADMIN_USERNAME"] = userdata.get("ADMIN_USERNAME")
env["ADMIN_PASSWORD"] = userdata.get("ADMIN_PASSWORD")

# Start Streamlit with the secrets available to the app
process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    env=env
)

# Wait for Streamlit to start
time.sleep(5)

# Create a new public URL
public_url = ngrok.connect(8501)

print("Your updated application URL:")
print(public_url)


Your updated application URL:
NgrokTunnel: "https://pregame-study-contend.ngrok-free.dev" -> "http://localhost:8501"


In [1]:
import os

if os.path.exists("app.py"):
    print("Great! Your app.py file exists.")
else:
    print("app.py was not found.")

Great! Your app.py file exists.


In [2]:

with open("app.py", "r") as file:
    code = file.read()

print(code)


import os
import sqlite3
import hashlib
import hmac
import jwt
import streamlit as st
from datetime import datetime, timedelta, timezone

DB_NAME = "users.db"
JWT_SECRET = os.environ.get("JWT_SECRET")

if not JWT_SECRET:
    st.error("JWT_SECRET is missing. Restart the app using the setup cell.")
    st.stop()

st.set_page_config(
    page_title="Infosys Authentication System",
    page_icon="🔐",
    layout="centered"
)

def connect_db():
    return sqlite3.connect(DB_NAME)

def create_table():
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                email TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                salt TEXT NOT NULL
            )
        """)

def hash_password(password, salt):
    return hashlib.pbkdf2_hmac(
        "sha256",
        password.encode(),
        bytes.fromhex(salt),
   

In [3]:

with open("app.py", "r") as file:
    code = file.read()

print(code)


import os
import sqlite3
import hashlib
import hmac
import jwt
import streamlit as st
from datetime import datetime, timedelta, timezone

DB_NAME = "users.db"
JWT_SECRET = os.environ.get("JWT_SECRET")

if not JWT_SECRET:
    st.error("JWT_SECRET is missing. Restart the app using the setup cell.")
    st.stop()

st.set_page_config(
    page_title="Infosys Authentication System",
    page_icon="🔐",
    layout="centered"
)

def connect_db():
    return sqlite3.connect(DB_NAME)

def create_table():
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                email TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                salt TEXT NOT NULL
            )
        """)

def hash_password(password, salt):
    return hashlib.pbkdf2_hmac(
        "sha256",
        password.encode(),
        bytes.fromhex(salt),
   

In [1]:

%%writefile app.py
import os
import sqlite3
import hashlib
import hmac
import secrets
import smtplib
import jwt
import streamlit as st
from email.message import EmailMessage
from datetime import datetime, timedelta, timezone

DB_NAME = "users.db"
JWT_SECRET = os.environ.get("JWT_SECRET")
EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD")

st.set_page_config(
    page_title="Infosys Authentication System",
    page_icon="🔐",
    layout="centered"
)

if not JWT_SECRET:
    st.error("JWT_SECRET is missing. Restart the app using the Colab launcher.")
    st.stop()

def connect_db():
    return sqlite3.connect(DB_NAME)

def create_table():
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                email TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                salt TEXT NOT NULL
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS password_otps (
                email TEXT PRIMARY KEY,
                otp_hash TEXT NOT NULL,
                expires_at TEXT NOT NULL,
                attempts INTEGER NOT NULL DEFAULT 0
            )
        """)

def hash_password(password, salt):
    return hashlib.pbkdf2_hmac(
        "sha256", password.encode(),
        bytes.fromhex(salt), 200000
    ).hex()

def create_token(username):
    return jwt.encode(
        {
            "username": username,
            "exp": datetime.now(timezone.utc) + timedelta(hours=1)
        },
        JWT_SECRET,
        algorithm="HS256"
    )

def get_logged_in_user():
    token = st.session_state.get("jwt_token")
    if not token:
        return None
    try:
        return jwt.decode(
            token, JWT_SECRET, algorithms=["HS256"]
        )["username"]
    except jwt.InvalidTokenError:
        st.session_state.pop("jwt_token", None)
        return None

def send_otp(email, otp):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        raise ValueError("Email secrets are missing.")

    message = EmailMessage()
    message["Subject"] = "Password Reset OTP"
    message["From"] = EMAIL_ADDRESS
    message["To"] = email
    message.set_content(
        f"Your password reset OTP is {otp}.\n"
        "It expires in 5 minutes. Do not share this code."
    )

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.send_message(message)

create_table()

st.title("🔐 User Authentication System")
st.caption("Infosys Springboard Internship 7.0 | Batch 4 | Milestone 1")

logged_in_user = get_logged_in_user()

if logged_in_user:
    st.success(f"Welcome, {logged_in_user}!")
    st.subheader("User Dashboard")
    st.write("You have successfully logged in.")
    st.write("Your session uses a JWT token.")

    if st.button("Logout"):
        st.session_state.pop("jwt_token", None)
        st.rerun()

else:
    page = st.sidebar.selectbox(
        "Choose Page",
        ["Login", "Signup", "Forgot Password"]
    )

    if page == "Signup":
        st.subheader("Create an Account")
        username = st.text_input("Username")
        email = st.text_input("Email Address")
        password = st.text_input("Password", type="password")
        confirm = st.text_input("Confirm Password", type="password")

        if st.button("Sign Up"):
            username = username.strip()
            email = email.strip().lower()

            if not username or not email or not password:
                st.error("Please fill in all fields.")
            elif "@" not in email or "." not in email.split("@")[-1]:
                st.error("Enter a valid email address.")
            elif len(password) < 8:
                st.error("Password must be at least 8 characters.")
            elif password != confirm:
                st.error("Passwords do not match.")
            else:
                salt = os.urandom(16).hex()
                password_hash = hash_password(password, salt)
                try:
                    with connect_db() as conn:
                        conn.execute(
                            """INSERT INTO users
                            (username, email, password_hash, salt)
                            VALUES (?, ?, ?, ?)""",
                            (username, email, password_hash, salt)
                        )
                    st.success("Account created. Please log in.")
                except sqlite3.IntegrityError:
                    st.error("Username or email is already registered.")

    elif page == "Login":
        st.subheader("Login to Your Account")
        username = st.text_input("Username")
        password = st.text_input("Password", type="password")

        if st.button("Login"):
            with connect_db() as conn:
                user = conn.execute(
                    """SELECT password_hash, salt FROM users
                    WHERE username = ?""",
                    (username.strip(),)
                ).fetchone()

            if user and hmac.compare_digest(
                hash_password(password, user[1]), user[0]
            ):
                st.session_state["jwt_token"] = create_token(username.strip())
                st.rerun()
            else:
                st.error("Incorrect username or password.")

    elif page == "Forgot Password":
        st.subheader("Reset Password with Email OTP")
        email = st.text_input("Registered Email Address").strip().lower()

        if st.button("Send OTP"):
            if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
                st.error("Email settings are missing. Restart the app.")
            else:
                with connect_db() as conn:
                    user = conn.execute(
                        "SELECT id FROM users WHERE email = ?", (email,)
                    ).fetchone()

                if not user:
                    st.error("No account found for that email.")
                else:
                    otp = f"{secrets.randbelow(1000000):06d}"
                    otp_hash = hashlib.sha256(otp.encode()).hexdigest()
                    expires = (
                        datetime.now(timezone.utc) + timedelta(minutes=5)
                    ).isoformat()

                    try:
                        send_otp(email, otp)
                        with connect_db() as conn:
                            conn.execute(
                                """INSERT OR REPLACE INTO password_otps
                                (email, otp_hash, expires_at, attempts)
                                VALUES (?, ?, ?, 0)""",
                                (email, otp_hash, expires)
                            )
                        st.session_state["reset_email"] = email
                        st.success("OTP sent to your registered email.")
                    except Exception:
                        st.error("Could not send OTP. Check Gmail settings.")

        reset_email = st.session_state.get("reset_email")

        if reset_email:
            st.info(f"Enter the OTP sent to: {reset_email}")
            entered_otp = st.text_input("6-digit OTP")
            new_password = st.text_input("New Password", type="password")
            confirm_new = st.text_input(
                "Confirm New Password", type="password"
            )

            if st.button("Reset Password"):
                with connect_db() as conn:
                    record = conn.execute(
                        """SELECT otp_hash, expires_at, attempts
                        FROM password_otps WHERE email = ?""",
                        (reset_email,)
                    ).fetchone()

                if not record:
                    st.error("Please request a new OTP.")
                elif record[2] >= 5:
                    st.error("Too many attempts. Request a new OTP.")
                elif datetime.now(timezone.utc) >= datetime.fromisoformat(
                    record[1]
                ):
                    st.error("OTP expired. Request a new one.")
                elif not hmac.compare_digest(
                    hashlib.sha256(entered_otp.encode()).hexdigest(),
                    record[0]
                ):
                    with connect_db() as conn:
                        conn.execute(
                            """UPDATE password_otps SET attempts = attempts + 1
                            WHERE email = ?""",
                            (reset_email,)
                        )
                    st.error("Incorrect OTP.")
                elif len(new_password) < 8:
                    st.error("Password must be at least 8 characters.")
                elif new_password != confirm_new:
                    st.error("Passwords do not match.")
                else:
                    new_salt = os.urandom(16).hex()
                    new_hash = hash_password(new_password, new_salt)

                    with connect_db() as conn:
                        conn.execute(
                            """UPDATE users SET password_hash = ?, salt = ?
                            WHERE email = ?""",
                            (new_hash, new_salt, reset_email)
                        )
                        conn.execute(
                            "DELETE FROM password_otps WHERE email = ?",
                            (reset_email,)
                        )

                    st.session_state.pop("reset_email", None)
                    st.success("Password reset successful. Please log in.")


Writing app.py


In [3]:
import sys
import subprocess

# Ensure required packages are installed
for package in ["pyngrok", "streamlit", "PyJWT"]:
    try:
        __import__(package if package != "PyJWT" else "jwt")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

from google.colab import userdata
from pyngrok import ngrok
import os
import time

try:
    process.terminate()
    process.wait(timeout=5)
except Exception:
    pass

ngrok.kill()

# Ensure ngrok auth token is configured
authtoken = userdata.get("NGROK_AUTHTOKEN")
if authtoken:
    ngrok.set_auth_token(authtoken)

env = os.environ.copy()
env["JWT_SECRET"] = userdata.get("JWT_SECRET")
env["EMAIL_ADDRESS"] = userdata.get("EMAIL_ADDRESS")
env["EMAIL_PASSWORD"] = userdata.get("EMAIL_PASSWORD")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    env=env
)

time.sleep(5)
public_url = ngrok.connect(8501)
print("Open your updated application:")
print(public_url)

Open your updated application:
NgrokTunnel: "https://pregame-study-contend.ngrok-free.dev" -> "http://localhost:8501"


In [4]:

# Add Admin Login to the existing navigation
with open("app.py", "r") as file:
    code = file.read()

old_options = '["Login", "Signup", "Forgot Password"]'
new_options = '["Login", "Signup", "Forgot Password", "Admin Login"]'

if old_options not in code:
    print("Navigation text not found. Please tell me before changing the code.")
else:
    code = code.replace(old_options, new_options, 1)

    old_branch = '    if page == "Signup":'
    new_branch = '''    if page == "Admin Login":
        st.subheader("Admin Login")

        if st.session_state.get("admin_logged_in", False):
            st.success("Admin authenticated")
            st.subheader("Registered Users")

            with connect_db() as conn:
                users = conn.execute(
                    "SELECT username, email FROM users ORDER BY username"
                ).fetchall()

            if users:
                st.dataframe(
                    [{"Username": u, "Email": e} for u, e in users],
                    use_container_width=True
                )
            else:
                st.info("No registered users found.")

            if st.button("Admin Logout"):
                st.session_state.pop("admin_logged_in", None)
                st.rerun()

        else:
            admin_username = st.text_input("Admin Username")
            admin_password = st.text_input(
                "Admin Password", type="password"
            )

            if st.button("Admin Login"):
                from google.colab import userdata

                expected_username = userdata.get("ADMIN_USERNAME")
                expected_password = userdata.get("ADMIN_PASSWORD")

                if (
                    admin_username == expected_username
                    and hmac.compare_digest(
                        admin_password, expected_password
                    )
                ):
                    st.session_state["admin_logged_in"] = True
                    st.rerun()
                else:
                    st.error("Invalid admin credentials.")

    elif page == "Signup":'''

    if old_branch not in code:
        print("Signup section not found. No changes saved.")
    else:
        code = code.replace(old_branch, new_branch, 1)
        with open("app.py", "w") as file:
            file.write(code)
        print("Admin Login code added successfully.")


Admin Login code added successfully.


In [5]:

env["ADMIN_USERNAME"] = userdata.get("ADMIN_USERNAME")
env["ADMIN_PASSWORD"] = userdata.get("ADMIN_PASSWORD")


In [7]:

with open("app.py", "r") as file:
    code = file.read()

old = '''                from google.colab import userdata

                expected_username = userdata.get("ADMIN_USERNAME")
                expected_password = userdata.get("ADMIN_PASSWORD")'''

new = '''                expected_username = os.environ.get("ADMIN_USERNAME")
                expected_password = os.environ.get("ADMIN_PASSWORD")'''

if old in code:
    code = code.replace(old, new, 1)

    with open("app.py", "w") as file:
        file.write(code)

    print("Admin Login code fixed successfully!")
else:
    print("The exact code was not found. Do not change anything else yet.")


Admin Login code fixed successfully!


In [9]:
env["ADMIN_USERNAME"] = userdata.get("ADMIN_USERNAME")
env["ADMIN_PASSWORD"] = userdata.get("ADMIN_PASSWORD")